# Lesson 1.2 - Objects and the Pipeline

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain why PowerShell uses objects instead of plain text
- Filter and reshape command output with the pipeline
- Export the results of a pipeline to a file

**Project file(s) produced:** `scripts/lesson-1.2-objects-pipeline.ps1` and `artifacts/process-report.csv`

## Concepts

Think of an object as a container with labeled fields. A process object might have a name, an id, CPU usage, and memory usage. Because the data is structured, you can sort it, filter it, and export it without having to guess where values appear in a text blob.

The pipeline moves objects from one command to the next. In PowerShell, the next command can decide which properties to keep, which rows to ignore, and how the final output should look.

Reference: [PowerShell Documentation](https://learn.microsoft.com/en-us/powershell/)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap

lesson_dir = Path('study-plan/topic-powershell/phase-1-foundations')
artifact_dir = lesson_dir / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

def run_powershell(script_text: str, script_name: str) -> int:
    script_path = lesson_dir / script_name
    script_path.write_text(script_text, encoding='utf-8')
    executable = shutil.which('pwsh') or shutil.which('powershell')
    if executable:
        result = subprocess.run([executable, '-NoLogo', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(script_path)], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        return result.returncode
    print(f'{script_path.name} written. Run it in PowerShell to see the live output.')
    return 0

## Code-Along: Step 1 - Shape process data

Here you collect a small set of process properties and keep only the columns that matter for a quick operational view. This is the PowerShell equivalent of turning raw data into a concise report.

In [ ]:
script = textwrap.dedent(r'''
$processReport = Get-Process | Select-Object -First 10 Name, Id, CPU, WorkingSet
$processReport | Format-Table -AutoSize
''')
run_powershell(script, 'lesson-1.2-shape-process-data.ps1')

## Code-Along: Step 2 - Export a pipeline result

Once data is structured, you can save it for another tool. CSV is a simple interchange format that most reporting tools and spreadsheet apps understand.

In [ ]:
script = textwrap.dedent(r'''
$csvPath = Join-Path $PSScriptRoot 'artifacts/process-report.csv'
$processReport = Get-Process | Sort-Object CPU -Descending | Select-Object -First 10 Name, Id, CPU, WorkingSet
$processReport | Export-Csv -Path $csvPath -NoTypeInformation
Get-Content $csvPath | Select-Object -First 5
''')
run_powershell(script, 'lesson-1.2-export-report.ps1')

## Exercise

Change the report so it keeps the top 5 processes by CPU and also includes the process path when that property is available.

**Your task:**
1. Update the property list.
2. Change the number of rows exported.
3. Re-run the notebook and confirm the CSV still writes successfully.

**Expected result:** a smaller CSV with one additional useful property.

In [ ]:
# TODO: add ProcessPath to the exported report and keep only the top 5 rows
exercise_script = textwrap.dedent(r'''
Write-Host "TODO: update the pipeline report"
''')
run_powershell(exercise_script, 'lesson-1.2-exercise.ps1')

## Key Takeaways

- Pipeline output is structured, so downstream commands can work with properties directly.
- `Select-Object`, `Sort-Object`, and `Export-Csv` are common reporting tools.
- A pipeline can become a repeatable operational report with only a few lines of code.

---

**Next:** [Lesson 1.3 - Variables, Collections, and Providers](./lesson-1.3-variables-collections-providers.ipynb)